# Training Models for NER Deliverable
Members: Miki, Jose, Lluc

In [1]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'utils'))

import utils
import sklearn_crfsuite
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

## 1. Load Data

In [2]:
train_sents = utils.load_data('data/train_data_ner.csv')
test_sents = utils.load_data('data/test_data_ner.csv')
print(f"Loaded {len(train_sents)} training sentences.")

Loaded 38366 training sentences.


## 2. CRF Model Training

In [3]:
X_train = [utils.sent2features(s) for s in train_sents]
y_train = [utils.sent2labels(s) for s in train_sents]

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)

# Save CRF model
with open('fitted_models/crf_model.pkl', 'wb') as f:
    pickle.dump(crf, f)
print("CRF model saved.")

CRF model saved.


## 3. Deep Learning Model (BiLSTM)

In [4]:
# Prepare vocabulary and tag mappings
word_to_ix = {"<PAD>": 0, "<UNK>": 1}
for sent in train_sents:
    for word, tag in sent:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

tag_to_ix = {"<PAD>": 0}
for sent in train_sents:
    for word, tag in sent:
        if tag not in tag_to_ix:
            tag_to_ix[tag] = len(tag_to_ix)

ix_to_tag = {v: k for k, v in tag_to_ix.items()}

with open('fitted_models/dl_mappings.pkl', 'wb') as f:
    pickle.dump({'word_to_ix': word_to_ix, 'tag_to_ix': tag_to_ix, 'ix_to_tag': ix_to_tag}, f)

class NERDataset(Dataset):
    def __init__(self, sents, word_to_ix, tag_to_ix, max_len=50):
        self.sents = sents
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix
        self.max_len = max_len
        
    def __len__(self):
        return len(self.sents)
    
    def __getitem__(self, idx):
        sent = self.sents[idx]
        X = [self.word_to_ix.get(w, self.word_to_ix["<UNK>"]) for w, t in sent][:self.max_len]
        y = [self.tag_to_ix[t] for w, t in sent][:self.max_len]
        
        # Padding
        X = X + [0] * (self.max_len - len(X))
        y = y + [0] * (self.max_len - len(y))
        
        return torch.tensor(X), torch.tensor(y)

train_dataset = NERDataset(train_sents, word_to_ix, tag_to_ix)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=50, hidden_dim=50):
        super(BiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.hidden2tag = nn.Linear(hidden_dim * 2, tagset_size)
        
    def forward(self, x):
        embeds = self.embedding(x)
        lstm_out, _ = self.lstm(embeds)
        tag_space = self.hidden2tag(lstm_out)
        return tag_space

model = BiLSTM(len(word_to_ix), len(tag_to_ix))
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Training BiLSTM...")
for epoch in range(5):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs.view(-1, outputs.shape[-1]), batch_y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), 'fitted_models/bilstm_model.pth')
print("BiLSTM model saved.")

Training BiLSTM...
Epoch 1, Loss: 0.4663
Epoch 2, Loss: 0.1909
Epoch 3, Loss: 0.1402
Epoch 4, Loss: 0.1141
Epoch 5, Loss: 0.0971
BiLSTM model saved.
